**Tabla de contenido**

- [Introducción](#Introduccio)
- [Librerías](#Librerias)
- [Preprocesamiento](#Preprocesamiento)
- [Preparación de los datos para modelado](#Preparacion-de-los-datos-para-modelado)
- [Modelado](#Modelado)

# Introduccion

Tu tarea consiste en crear un clasificador binario que prediga si un comentario de Reddit infringe una norma específica. El conjunto de datos procede de una gran colección de comentarios moderados, con una serie de normas de subreddit, tonos y expectativas de la comunidad.

`dataset`
- **body** - el texto del comentario
- **rule** - la regla que se considera que infringe el comentario
- **subreddit** - el foro en el que se hizo el comentario
- **positive_example_{1,2}** - ejemplos de comentarios que infringen la regla
- **negative_example_{1,2}** - ejemplos de comentarios que no infringen la regla
- **rule_violation** - el objetivo binario


# Librerias

In [1]:
import os
import pandas as pd
import re

In [2]:
file_path = lambda file: os.path.join(os.getcwd(),'data/Agile Community Rules Classification',file)
train = pd.read_csv(file_path('train.csv'))
#train = train.set_index('row_id', drop=True)
#pd.set_option('display.max_colwidth', None)  # Mostrar todo el contenido de las celd
train.head(2)

,row_id,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,rule_violation
0,0,Banks don't want you to know this! Click here ...,"No Advertising: Spam, referral links, unsolici...",Futurology,If you could tell your younger self something ...,hunt for lady for jack off in neighbourhood ht...,Watch Golden Globe Awards 2017 Live Online in ...,"DOUBLE CEE x BANDS EPPS - ""BIRDS""\n\nDOWNLOAD/...",0
1,1,SD Stream [ ENG Link 1] (http://www.sportsstre...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,[I wanna kiss you all over! Stunning!](http://...,LOLGA.COM is One of the First Professional Onl...,#Rapper \n🚨Straight Outta Cross Keys SC 🚨YouTu...,[15 Amazing Hidden Features Of Google Search Y...,0


In [3]:
print(train['positive_example_1'][4])

 wow!! amazing reminds me of the old days.Well Do you desire a great spell caster and a herbal doctor to help you solve any problem you are going through? i am a proud testimony of what king favour solution temple has offered me. Contact him now at kingfavoursolutiontemple@yahoo.com You will be the next to testify.bye everyone


# Preprocesamiento

Vamos a prepar los datos para el modelo Bertweet. BERTweet es un modelo de lenguaje basado en la arquitectura BERT (Bidirectional Encoder Representations from Transformers), pero específicamente entrenado en tweets (textos de Twitter) en ingles. Está optimizado para lenguaje informal, lo que incluye jerga de redes sociales, hashtags, emoticonos, menciones (@) y ortografía no estándar (ej: "loooove"). Tiene un Tokenizador adaptado: Maneja mejor palabras repetidas ("goooool"), contracciones ("don't" → "do n't") y palabras concatenadas ("NewYork").

Este modelo está diseñado para tareas de Procesamiento de Lenguaje Natural (NLP) en redes sociales, como:

1. `Clasificación de Texto`

- Análisis de sentimiento (ej.: ¿Es un tweet positivo, negativo o neutro?).
- Detección de hate speech, spam o bullying.
- Identificación de noticias falsas (fake news) en redes sociales.

2. `Extracción de Información`

- Named Entity Recognition (NER): Identificar personas, lugares, etc., en tweets.
- Detección de temas (topic modeling) en conversaciones de Twitter.

3. `Aplicaciones Específicas`

- Moderación automática de contenido en plataformas sociales.
- Respuesta a preguntas (QA) en contextos informales.
- Generación de texto (aunque no es su enfoque principal).

Esto implica que el preprocesamiento de los textos debe realizarse de la siguiente forma:

1. Reemplazar las URL por la abreviatura `[URL]`.
2. Reemplazar las mensiones de usarios por la abreviatura `[USER]`
3. Los `Hashtag` deben dejarse tal cual como están.
4. Los emojis deben dejarse ya que ayudan al modelo a entender tono emosional o sarcasmo.
5. Se deben eliminar múltiples espacios, tabs o saltos de linea innecesarios.
6. No convertir a mayúscula o minúscula, ni eliminar los signos de puntuación. Este modelo no distigue entre mayúscula/minúscula.

In [4]:
from urlextract import URLExtract

def cleantext_toBERTweet(text):
    text = re.sub(r"\s+"," ", text).strip()                     # reemplaza múltiples espacios, tabs o saltos de línea por un solo espacio
    email_pattern = r'\b([A-Za-z0-9._%+-]+)\s*(?:@|\[at\]|\(at\)|arroba)\s*([A-Za-z0-9.-]+)\s*(?:\.|\[dot\]|\(dot\)|punto)\s*([A-Za-z]{2,})\b'
    text = re.sub(email_pattern,'[EMAIL]',text)                 # reemplaza correo electrónicos a [EMAIL]
    phone_pattern = r'(?<!\w)(?:\+?\d{1,3}|\(\+?\d{1,3}\))?(?:[-. /]?\d{2,4}){2,5}(?:[-. /]?\d{2,})\b(?:[ ]*(?:ext|xtn|x|#)[ ]*\d{1,6})?(?!\w)'
    text = re.sub(phone_pattern, '[PHONE]', text)               # Reemplaza números de teléfonos por [PHONE]

    # Reemplazo de URLs estándar detectadas por URLExtract
    extractor = URLExtract()
    urls = extractor.find_urls(text)
    for url in urls:
        text = text.replace(url, '[URL]')  
    text = re.sub(r"@\w+", " [USER] ", text)                    # reemplaza usuarios por [USER]

    # Patrón “raro” en url (/p/... .xxx)
    pattern_url_raro =  r"/[A-Za-z]/[\w-]+\.[A-Za-z]{2,4}\b"
    text = re.sub(pattern_url_raro, "[URL]", text)
    # Cualquier HTTP/HTTPS
    pattern_any_url = r"(?:https?://|://)[^\s]+"
    text = re.sub(pattern_any_url, "[URL]", text)
    # patrones raros
    pattern_scheme = r"\b[a-z][\w+.-]*://[^\s]+\b"
    text = re.sub(pattern_scheme, "[URL]", text)

    # “come” todas las aperturas de paréntesis o corchetes adyacentes antes de un token del tipo […]
    pattern =  r'([(\[])\[URL\]([)\]])'  # Captura ( [URL] ) o [ [URL] ]
    text = re.sub(pattern,'[URL]',text)
    text = re.sub(r"\*", "", text)
    text = re.sub(r'\/','',text)
    return text

Veamos ahora como quedan los texto, para esto sacaremos muestras aleatorias y las limpiaremos, esto con el fin de saber que todo está ok.

In [5]:
#train = train.set_index('row_id', drop=True)
pd.set_option('display.max_colwidth', None)  # Mostrar todo el contenido de las celd
muestra = train['body'].sample(n=10)
muestra.head(10)

454         [Euro Cup 2016 English Stream 1](http://www.sportshub2016.ml/p/async-srcpagead2.html) | |  [Euro Cup 2016 English Stream 2](http://www.sportshub2016.ml/p/adsbygoogle-window.html)  | |  [Euro Cup 2016 French Stream 2](http://www.sportshub2016.ml/p/adsbygoogle-window.html)  | |  [Euro Cup 2016 German Stream](http://www.sportshub2016.ml/p/adsbygoogle-window_14.html)
903                                                                                                                                                                                                                                                                                           He is breaking the law? You should report him so he can be deported then come back legally.
1215                                                                                                                  fair enough, current civil and potentially criminal. The potential for jail is definitely there though I mean I dont see how r

In [6]:
muestra = muestra.apply(cleantext_toBERTweet)
muestra.head(10)

454                                                                                                                                                                                                         [Euro Cup 2016 English Stream 1][URL] | | [Euro Cup 2016 English Stream 2][URL] | | [Euro Cup 2016 French Stream 2][URL] | | [Euro Cup 2016 German Stream][URL]
903                                                                                                                                                                                                                                                                             He is breaking the law? You should report him so he can be deported then come back legally.
1215                                                                                                    fair enough, current civil and potentially criminal. The potential for jail is definitely there though I mean I dont see how running an entire company built around crim

Podemos ver que la función parece funcionar correctamente. Apliquemosla al set de datos.

In [7]:
df_train_ob = train.select_dtypes(['object'])
for col in df_train_ob.columns:
    df_train_ob[col]=df_train_ob[col].apply(cleantext_toBERTweet)
df_train_ob.head()

,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2
0,Banks don't want you to know this! Click here to know more!,"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",Futurology,"If you could tell your younger self something different about sex, what would that be? i AM IN A CONTEST TO WIN FUNDING FOR MY SEX POSITIVE FILM: VOTE HERE: [URL]",hunt for lady for jack off in neighbourhood [URL],Watch Golden Globe Awards 2017 Live Online in HD Coverage without ADS (VIP STREAMS) = HD STREAM QUALITY >>> [WATCH LINK1][URL] = HD BROADCASTING QUALITY >>> [WATCH LINK1][URL] = Mobile Compatibility: YES = NO ADS | NO ADS | ADS =,"DOUBLE CEE x BANDS EPPS - ""BIRDS"" DOWNLOADSTREAM: [URL]"
1,SD Stream [ ENG Link 1] [URL],"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",soccerstreams,[I wanna kiss you all over! Stunning!][URL],"[URL] is One of the First Professional Online Gold sites. By Now, As A Game Gold Seller, we've over more than 5 yrs Of Experience And Can Pass That On To Our Customers.","#Rapper 🚨Straight Outta Cross Keys SC 🚨YouTube Search Beanie 864 Click Link BELOW To Hear Hit Single ""Ah Man"" Beanie 864 FEAT King Kota (King Kota Is Only 15!) Lit 🌡🔥👍💵💯Fr Fr [URL]","[15 Amazing Hidden Features Of Google Search You Probably Don’t Know]([URL] No one would argue the fact that Google is one of the most useful si[URL]-amazing-hidden-features-of-google-search-you-probably-dont-knowtes on the Internet. Unfortunately, most people only use about...?utm_source=reddit&utm_campaign=samreen&utm_medium=cpc)"
2,Lol. Try appealing the ban and say you won't do it again.,No legal advice: Do not offer or request legal advice.,pcmasterrace,"Don't break up with him or call the cops. If you are willing to get beat up by him to stay with him, he is obviously a real winner and you know it, so you shouldn't leave him.",It'll be dismissed: [URL] The first amendment law here is SUPER settled.,Where is there a site that still works where you can jump the GPS. Is there a FAQ to do this with iPhone or Mac?,"Because this statement of his is true. It isn't freedom of the press, it's libel. And because of this, your post serves as a promotion for Trump. Reported."
3,she will come your home open her legs with and you [URL],"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",sex,Selling Tyrande codes for 3€ to paypal. PM. [URL],tight pussy watch for your cock get her at this point [URL],NSFW(obviously) [URL],Good News ::Download WhatsApp 2.16.230 APK for Android – Latest Version
4,code free tyrande --->>> [Imgur][URL] for you and your friend 2 codes for 4 dollars [URL] 2$... buy one directly from here: [URL],"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",hearthstone,wow!! amazing reminds me of the old days.Well Do you desire a great spell caster and a herbal doctor to help you solve any problem you are going through? i am a proud testimony of what king favour solution temple has offered me. Contact him now at [EMAIL] You will be the next to testify.bye everyone,seek for lady for sex in around [URL],must be watch movie [URL],"We're streaming Pokemon Veitnamese Crystal RIGHT NOW, come watch [URL]"


Perfecto!. Con esto hemos reemplazados las URL a `[URL]`, los email a `[EMAIL]`, los numéros de telefono a [PHONE]. Ahora lo que sigue es eliminar el ruido que no es útil para el modelo. Por ejemplo:

- Cadenas sin letras (solo números + símbolos, ej: 43567&%^*).
- Símbolos repetidos o combinaciones sin sentido (ej: !!??, %%%, --==).
- Caracteres especiales sueltos (ej: &, ^, * si no están pegados a palabras).

In [8]:
def eliminar_ruido(text):
    pattern = r'FindSexToday\s*\.\s*com'
    text = re.sub(pattern, '[URL]', text, flags=re.IGNORECASE)
    text = re.sub(r'\.{1,}', '.', text)
    text = re.sub(r'\!{1,}','!',text) # Cualquier secuencia de 4 o más ! se reemplazará por ! (tres signos de exclamación).
    text = re.sub(r'\={2,}', '=', text)
    text = re.sub(r'\-{2,}','-',text)
    text = re.sub(r'\<{2,}','<',text)
    text = re.sub(r'\>{2,}','>',text)
    text = re.sub(r'\. \.\.\.', '.', text)  # Reemplaza ". ..." por "."
    text = re.sub(r'\.{2,}', '.', text) # reemplaza mas de un punto consecutivo por un punto
    text = re.sub(r'(\[URL\]\s*){2,}', '[URL]', text) 
    text = re.sub(r'\?{2,}','?',text)
    text = re.sub(r'(?<=[a-zA-Z])[.,](?=[a-zA-Z])', r'\g<0> ', text)  # gregar un espacio después de los signos de puntuación básicos 
    text = re.sub(r'\s+([.,;])', r'\1', text)  # Elimina espacios antes los signo ,.;
    text = re.sub(r'\|', '', text)
    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'\b\d{1,3}-\d{2,3}-\d{2,4}(?:-\d{2,4})?\b', '[PHONE]', text)
    text = text.strip()
    return text
    

In [9]:
muestra = df_train_ob['body'].sample(n=10)
muestra.head(10)

618     Rapists and domestic abusers can get visitation, so unless she is specifically hurting the children, she will most likely get partial custody or visitation. Even if she has harmed the children, she could most likely get supervised visitation.
2000                                                                                                                                                                                        Send a tip to the authorities, this type of comment is illegal
1684                                                                                                                           There's a sign-up offer through Jan. 11, 2017: [$150 sign-up bonus for purchasing at least $500 in the first 3 months][URL]
1139                                                          Hi! Need a way to earn extra money for food, necessities, and fun? Start earning money easily and more efficiently from your smart device or computer by generating web traffic today! [U

In [10]:
muestra = muestra.apply(eliminar_ruido)
muestra.head(10)

618     Rapists and domestic abusers can get visitation, so unless she is specifically hurting the children, she will most likely get partial custody or visitation. Even if she has harmed the children, she could most likely get supervised visitation.
2000                                                                                                                                                                                        Send a tip to the authorities, this type of comment is illegal
1684                                                                                                                           There's a sign-up offer through Jan. 11, 2017: [$150 sign-up bonus for purchasing at least $500 in the first 3 months][URL]
1139                                                          Hi! Need a way to earn extra money for food, necessities, and fun? Start earning money easily and more efficiently from your smart device or computer by generating web traffic today! [U

Eliminemos el ruido con la función creada.

In [11]:
df_train_ob = df_train_ob.select_dtypes(['object'])
for col in df_train_ob.columns:
    df_train_ob[col]=df_train_ob[col].apply(eliminar_ruido)
df_train_ob.head()

,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2
0,Banks don't want you to know this! Click here to know more!,"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",Futurology,"If you could tell your younger self something different about sex, what would that be? i AM IN A CONTEST TO WIN FUNDING FOR MY SEX POSITIVE FILM: VOTE HERE: [URL]",hunt for lady for jack off in neighbourhood [URL],Watch Golden Globe Awards 2017 Live Online in HD Coverage without ADS (VIP STREAMS) = HD STREAM QUALITY > [WATCH LINK1][URL] = HD BROADCASTING QUALITY > [WATCH LINK1][URL] = Mobile Compatibility: YES = NO ADS NO ADS ADS =,"DOUBLE CEE x BANDS EPPS - ""BIRDS"" DOWNLOADSTREAM: [URL]"
1,SD Stream [ ENG Link 1] [URL],"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",soccerstreams,[I wanna kiss you all over! Stunning!][URL],"[URL] is One of the First Professional Online Gold sites. By Now, As A Game Gold Seller, we've over more than 5 yrs Of Experience And Can Pass That On To Our Customers.","#Rapper 🚨Straight Outta Cross Keys SC 🚨YouTube Search Beanie 864 Click Link BELOW To Hear Hit Single ""Ah Man"" Beanie 864 FEAT King Kota (King Kota Is Only 15!) Lit 🌡🔥👍💵💯Fr Fr [URL]","[15 Amazing Hidden Features Of Google Search You Probably Don’t Know]([URL] No one would argue the fact that Google is one of the most useful si[URL]-amazing-hidden-features-of-google-search-you-probably-dont-knowtes on the Internet. Unfortunately, most people only use about.?utm_source=reddit&utm_campaign=samreen&utm_medium=cpc)"
2,Lol. Try appealing the ban and say you won't do it again.,No legal advice: Do not offer or request legal advice.,pcmasterrace,"Don't break up with him or call the cops. If you are willing to get beat up by him to stay with him, he is obviously a real winner and you know it, so you shouldn't leave him.",It'll be dismissed: [URL] The first amendment law here is SUPER settled.,Where is there a site that still works where you can jump the GPS. Is there a FAQ to do this with iPhone or Mac?,"Because this statement of his is true. It isn't freedom of the press, it's libel. And because of this, your post serves as a promotion for Trump. Reported."
3,she will come your home open her legs with and you [URL],"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",sex,Selling Tyrande codes for 3€ to paypal. PM. [URL],tight pussy watch for your cock get her at this point [URL],NSFW(obviously) [URL],Good News ::Download WhatsApp 2.16.230 APK for Android – Latest Version
4,code free tyrande -> [Imgur][URL] for you and your friend 2 codes for 4 dollars [URL] 2$. buy one directly from here: [URL],"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",hearthstone,wow! amazing reminds me of the old days. Well Do you desire a great spell caster and a herbal doctor to help you solve any problem you are going through? i am a proud testimony of what king favour solution temple has offered me. Contact him now at [EMAIL] You will be the next to testify. bye everyone,seek for lady for sex in around [URL],must be watch movie [URL],"We're streaming Pokemon Veitnamese Crystal RIGHT NOW, come watch [URL]"


# Preparacion de los datos para modelado

Dado que nuestro objetivo es entrenar el modelo BERTweet para que logre identificar si un comentario viola las reglas, debemos hacer lo siguiente:

`Tokens especiales: BERTweet usa los mismos tokens especiales que BERT:`

- [CLS] al inicio.

- [SEP] para separar segmentos.


In [12]:
df_train_ob['combined_text'] = (
    "[CLS] comment: " + df_train_ob['body'] + 
    " [SEP] rule: " + df_train_ob['rule'] + 
    " [SEP] positive examples: " + df_train_ob['positive_example_1'] + 
    " [SEP] negative examples: " + df_train_ob['negative_example_1'] +  
    " [SEP]"
)
df_train_ob['combined_text'][0]

"[CLS] comment: Banks don't want you to know this! Click here to know more! [SEP] rule: No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed. [SEP] positive examples: If you could tell your younger self something different about sex, what would that be? i AM IN A CONTEST TO WIN FUNDING FOR MY SEX POSITIVE FILM: VOTE HERE: [URL] [SEP] negative examples: Watch Golden Globe Awards 2017 Live Online in HD Coverage without ADS (VIP STREAMS) = HD STREAM QUALITY > [WATCH LINK1][URL] = HD BROADCASTING QUALITY > [WATCH LINK1][URL] = Mobile Compatibility: YES = NO ADS NO ADS ADS = [SEP]"

Perfecto, Ahora necesitamos tokenizar el texto y consultar cual es el token con mayor longitud

In [13]:
from transformers import BertweetTokenizer

# Cargar tokenizador
tokenizer = BertweetTokenizer.from_pretrained("vinai/bertweet-base")
# Función para contar tokens
def count_tokens(text):
    return len(tokenizer.tokenize(text))

# Aplicar a cada fila y guardar la longitud
df_train_ob['token_length'] = df_train_ob['combined_text'].apply(count_tokens)

In [14]:
df_train_ob.head(1)

,body,rule,subreddit,positive_example_1,positive_example_2,negative_example_1,negative_example_2,combined_text,token_length
0,Banks don't want you to know this! Click here to know more!,"No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed.",Futurology,"If you could tell your younger self something different about sex, what would that be? i AM IN A CONTEST TO WIN FUNDING FOR MY SEX POSITIVE FILM: VOTE HERE: [URL]",hunt for lady for jack off in neighbourhood [URL],Watch Golden Globe Awards 2017 Live Online in HD Coverage without ADS (VIP STREAMS) = HD STREAM QUALITY > [WATCH LINK1][URL] = HD BROADCASTING QUALITY > [WATCH LINK1][URL] = Mobile Compatibility: YES = NO ADS NO ADS ADS =,"DOUBLE CEE x BANDS EPPS - ""BIRDS"" DOWNLOADSTREAM: [URL]","[CLS] comment: Banks don't want you to know this! Click here to know more! [SEP] rule: No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed. [SEP] positive examples: If you could tell your younger self something different about sex, what would that be? i AM IN A CONTEST TO WIN FUNDING FOR MY SEX POSITIVE FILM: VOTE HERE: [URL] [SEP] negative examples: Watch Golden Globe Awards 2017 Live Online in HD Coverage without ADS (VIP STREAMS) = HD STREAM QUALITY > [WATCH LINK1][URL] = HD BROADCASTING QUALITY > [WATCH LINK1][URL] = Mobile Compatibility: YES = NO ADS NO ADS ADS = [SEP]",174


In [15]:
max_token_row = df_train_ob.loc[df_train_ob['token_length'].idxmax()]

print("Texto con más tokens:")
print(max_token_row['combined_text'])
print(f"\nNúmero de tokens: {max_token_row['token_length']}")

Texto con más tokens:
[CLS] comment: Don't get out of the house, he did nothing wrong. Kick the lying bitch wife out of the house or he risks losing it in the divorce. I don't know a lot about this but I do know that it's mentioned when ending a marriage it's bad to leave the home as somehow it can legally jeopardize who gets to stay in the divorce. Not that he would want to live there after anyway but he can at least sell the house, maybe to a nice swinging couple. [SEP] rule: No legal advice: Do not offer or request legal advice. [SEP] positive examples: Please see 'legal name fraud' which you can research if the American police state government hasn't blocked it yet! They cannot touch you if you reject the slave name handed to you when you were a new-born baby. The IRS is an agency of the IMF, a foreign entity. IRS rules apply to persons not humans. As a human, you are free from all government rules and regulations: but if you consent to represent the NAME then you are their servant

Perfecto, hay un texto que contiene 466 tokens. Esto supera el token máximo permitido por Bertweet, que es de 128. Esto indica que debo usar berttweet large.

# Modelado

Antes de entar a modelar, es necesario saber si las clases estan balanceadas.

In [16]:
train['rule_violation'].value_counts(normalize=True)*100

rule_violation
1    50.813208
0    49.186792
Name: proportion, dtype: float64

En teoria las clases están balanceadas.

`Los modelos de transformadores como DistilBERT no pueden recibir cadenas de texto sin procesar como entrada; en su lugar, asumen que el texto ha sido tokenizado y codificado como vectores numéricos`. `La tokenización es el paso de descomponer una cadena en las unidades  utilizadas en el modelo`. Pero antes de tokenizar creemos el dataframe que contiene lo que necesitamos.


In [17]:
df_train = pd.concat([df_train_ob['combined_text'],train['rule_violation']],axis=1)
df_train.head(1)

,combined_text,rule_violation
0,"[CLS] comment: Banks don't want you to know this! Click here to know more! [SEP] rule: No Advertising: Spam, referral links, unsolicited advertising, and promotional content are not allowed. [SEP] positive examples: If you could tell your younger self something different about sex, what would that be? i AM IN A CONTEST TO WIN FUNDING FOR MY SEX POSITIVE FILM: VOTE HERE: [URL] [SEP] negative examples: Watch Golden Globe Awards 2017 Live Online in HD Coverage without ADS (VIP STREAMS) = HD STREAM QUALITY > [WATCH LINK1][URL] = HD BROADCASTING QUALITY > [WATCH LINK1][URL] = Mobile Compatibility: YES = NO ADS NO ADS ADS = [SEP]",0


Perfecto, ahora necesitamos llevar estos datos a un formato adecuado.

Este código prepara los textos y sus etiquetas para que un modelo BERTweet pueda entrenarse en una tarea de clasificación (detección de violaciones de reglas, probablemente en tweets). Convierte texto a tensores, los organiza en un dataset compatible con PyTorch y deja todo listo para usar con Hugging Face Trainer

In [18]:
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
import torch, gc
# Tokenización
# 1. Cargar tokenizer (usando la versión fast si es compatible)
tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-large", use_fast=True)  # use_fast=True es más eficiente

# 2. Preprocesamiento y tokenización más eficiente
def tokenize_data(texts, max_length=360):
    return tokenizer(
        texts,
        truncation=True,
        padding='longest',  # Padding dinámico hasta la máxima longitud en el batch
        max_length=max_length,
        return_tensors="pt"  # Devuelve tensores PyTorch directamente
    )

# 3. Dataset optimizado
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        # Ya tenemos tensores, no necesitamos conversión
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
        
    def __len__(self):
        return len(self.labels)

# 4. Procesamiento más eficiente
# Primero dividimos (es más eficiente tokenizar después)
X_train, X_val, y_train, y_val = train_test_split(
    df_train['combined_text'].tolist(),
    df_train['rule_violation'].tolist(),
    test_size=0.2,
    random_state=42
)

# Tokenización con batches para mejor rendimiento
train_encodings = tokenize_data(X_train)
val_encodings = tokenize_data(X_val)

# Crear datasets
train_dataset = CustomDataset(train_encodings, y_train)
val_dataset = CustomDataset(val_encodings, y_val)


Perfecto, ahora lo que tenemos que hacer es 

In [19]:
from transformers import AutoModelForSequenceClassification
model_ckpt = "vinai/bertweet-large"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_labels = 2
model = (AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=num_labels).to(device))

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Para monitorear las métricas durante el entrenamiento, necesitamos definir una función compute_metrics() para el Entrenador. Esta función recibe un objeto EvalPrediction (que es una tupla con nombre con atributos de predicciones y label_ids) y debe devolver un diccionario que mapea el nombre de cada métrica a su valor. Para nuestra aplicación, calcularemos la puntuación F1 y la precisión del modelo de la siguiente manera:

In [20]:
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

In [21]:
from transformers import Trainer, TrainingArguments
batch_size = 2  # Redujimos de 8 a 4 (o incluso 2 si persiste el error)
model_name = f"{model_ckpt}-finetuned-violacion"

training_args = TrainingArguments(output_dir=model_name,
                                  num_train_epochs=2,
                                  learning_rate=2e-5,
                                  per_device_train_batch_size=batch_size,
                                  per_device_eval_batch_size=batch_size,
                                  weight_decay=0.01,
                                  eval_strategy="epoch",
                                  disable_tqdm=False,
                                  push_to_hub=False,
                                  log_level="error")

In [22]:
from transformers import DataCollatorWithPadding

# Asegúrate de que el tokenizer tiene una longitud máxima manejable
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)


In [23]:
del X_train, X_val
gc.collect(); torch.cuda.empty_cache()
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.722400,0.702810,0.455665,0.285273
2,0.695100,0.692516,0.544335,0.383726


TrainOutput(global_step=1624, training_loss=0.7095686438048414, metrics={'train_runtime': 901.5526, 'train_samples_per_second': 3.6, 'train_steps_per_second': 1.801, 'total_flos': 2126987722520640.0, 'train_loss': 0.7095686438048414, 'epoch': 2.0})